In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent   # nbconvert and Jupyter both start the kernel in notebooks/
assert (ROOT / "paths.py").is_file(), f"paths.py not found from {Path.cwd()}"
sys.path.insert(0, str(ROOT))

# Step 4: global-parameter fit with leave-one-block-out scoring

For each training block: fit (g_diff, f_ir, ti_scale) on the other training blocks, then score the held-out block. The benchmark (mean REMS ground temperature per 15-minute LTST bin and 30-degree Ls bin) is built inside the same folds. Residual = prediction - REMS.

- Fitting rows exclude placeholder-tau sols and sols 3763-3778; both are scored separately.
- The benchmark uses all training rows of the fold.
- Objective: plain least squares over fitting rows (no block weighting).

In [ ]:
# --- config ---
import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import paths
import thermal1d as t

SUSPECT_SOLS = (3763, 3778)
EXPECTED_TRAIN_BLOCKS = [0, 1, 2, 3, 5, 6, 7, 9, 10]
LS_BIN_DEG = 30
OUT = paths.OUT_DIR
OUT.mkdir(parents=True, exist_ok=True)
PROGRESS_LOG = OUT / "lobo_progress.log"
PROGRESS_LOG.write_text("")
print("grid points per parameter:", t.FIT_GRID_POINTS, "| Gauss-Newton steps:", t.FIT_GN_STEPS,
      "| bounds:", dict(zip(t.PARAM_NAMES, t.PARAM_BOUNDS.tolist())))

## Data and evaluator

In [ ]:
tbl = pd.DataFrame(t.load_rems_table(paths.DATA_DIR / "gale_rems_training.csv"))
assert (tbl.split_role == "train").all()
blocks = sorted(tbl.block.unique())
assert blocks == EXPECTED_TRAIN_BLOCKS, f"training blocks {blocks}"

per_sol_cols = ["ls_deg", "tau", "tau_is_placeholder", "row", "slope_deg", "aspect_deg",
                "albedo_mean", "ti_themis_tiu", "block"]
assert (tbl.groupby("sol")[per_sol_cols].nunique() == 1).all().all(), "per-sol inputs vary within a sol"
sols = tbl.groupby("sol")[per_sol_cols].first().reset_index()
sol_inputs = {"lat_deg": t.lat_from_row(sols.row.to_numpy()), "ls_deg": sols.ls_deg.to_numpy(),
              "tau": sols.tau.to_numpy(), "slope_deg": sols.slope_deg.to_numpy(),
              "aspect_deg": sols.aspect_deg.to_numpy(), "albedo": sols.albedo_mean.to_numpy(),
              "ti": sols.ti_themis_tiu.to_numpy()}
row_sol = pd.Series(np.arange(len(sols)), index=sols.sol)[tbl.sol].to_numpy()
row_step = t.bin_centre_step(tbl.bin.to_numpy())
evaluate = t.make_evaluator(sol_inputs, row_sol, row_step)

obs = tbl.ground_temp_k.to_numpy()
suspect = tbl.sol.between(*SUSPECT_SOLS).to_numpy()
placeholder = tbl.tau_is_placeholder.to_numpy()
fit_ok = ~suspect & ~placeholder
tbl["group"] = np.where(suspect, "suspect_terrain", np.where(placeholder, "placeholder_tau", "main"))
tbl["ltst_hour"] = (tbl.bin // 4).astype(int)
tbl["ls_bin"] = (tbl.ls_deg // LS_BIN_DEG * LS_BIN_DEG).astype(int)
print(len(sols), "sols,", len(tbl), "rows;", int(fit_ok.sum()), "rows usable for fitting")
print(tbl.group.value_counts().to_string())

## Folds

In [ ]:
def benchmark(train, test):
    """Mean REMS ground temperature per (bin, Ls bin) from `train`; per-bin mean where a cell is empty."""
    cell = train.groupby(["bin", "ls_bin"]).ground_temp_k.mean()
    by_bin = train.groupby("bin").ground_temp_k.mean()
    pred = cell.reindex(pd.MultiIndex.from_arrays([test.bin, test.ls_bin])).to_numpy().copy()
    empty = np.isnan(pred)
    pred[empty] = by_bin.reindex(test.bin[empty]).to_numpy()
    assert np.all(np.isfinite(pred)), "benchmark has no value for some LTST bin"
    return pred, int(empty.sum())


def fit_row(label, p, sse, at_bound, n_fit, n_fallback):
    return {"fold": label, **dict(zip(t.PARAM_NAMES, p)),
            **{f"{n}_at_bound": bool(a) for n, a in zip(t.PARAM_NAMES, at_bound)},
            "fit_rows": n_fit, "fit_rmse_k": np.sqrt(sse / n_fit), "bench_fallback_rows": n_fallback}


block = tbl.block.to_numpy()
tbl["baseline_k"] = np.nan
tbl["bench_k"] = np.nan
fits = []
t0 = time.time()
for b in blocks:
    held = block == b
    p, sse, at_bound = t.fit_global_params(evaluate, obs, fit_ok & ~held)
    n_fit = int((fit_ok & ~held).sum())
    tbl.loc[held, "baseline_k"] = evaluate(p)[held]
    bench, n_fallback = benchmark(tbl[~held], tbl[held])
    tbl.loc[held, "bench_k"] = bench
    fits.append(fit_row(f"block {b}", p, sse, at_bound, n_fit, n_fallback))
    line = (f"fold {b:2d}: " + ", ".join(f"{n}={v:.4f}{' (BOUND)' if a else ''}" for n, v, a in zip(t.PARAM_NAMES, p, at_bound))
            + f" | fit RMSE {np.sqrt(sse / n_fit):.2f} K | benchmark fallback rows {n_fallback} | {time.time() - t0:.0f} s")
    print(line)
    with open(PROGRESS_LOG, "a") as fh:   # nbconvert shows output only at the end
        fh.write(line + "\n")

p, sse, at_bound = t.fit_global_params(evaluate, obs, fit_ok)
fits.append(fit_row("all training blocks", p, sse, at_bound, int(fit_ok.sum()), 0))
fits = pd.DataFrame(fits)
fits.to_csv(OUT / "fit_params.csv", index=False)
assert tbl[["baseline_k", "bench_k"]].notna().all().all()
print(fits.round(4).to_string())

## Scores

In [ ]:
tbl["resid_base"] = tbl.baseline_k - tbl.ground_temp_k
tbl["resid_bench"] = tbl.bench_k - tbl.ground_temp_k


def score(g):
    return pd.Series({"n": len(g),
                      "baseline_rmse_k": np.sqrt((g.resid_base ** 2).mean()), "bench_rmse_k": np.sqrt((g.resid_bench ** 2).mean()),
                      "baseline_bias_k": g.resid_base.mean(), "bench_bias_k": g.resid_bench.mean()})


scored = tbl[tbl.group != "placeholder_tau"]        # block 10 is all suspect_terrain and is its own block
by_block = scored.groupby("block").apply(score, include_groups=False)
main = tbl[tbl.group == "main"]
by_group = tbl.groupby("group").apply(score, include_groups=False)
by_hour = main.groupby("ltst_hour").apply(score, include_groups=False)
by_ls = main.groupby("ls_bin").apply(score, include_groups=False)

pd.options.display.float_format = "{:.2f}".format
print("Per block (placeholder-tau rows excluded; block 10 = sols 3763-3778):")
print(by_block.to_string())
for label, bb in (("with sols 3763-3778", by_block), ("without sols 3763-3778", by_block.drop(index=10))):
    print(f"macro RMSE {label}: baseline {bb.baseline_rmse_k.mean():.2f} K, benchmark {bb.bench_rmse_k.mean():.2f} K "
          f"({len(bb)} blocks)")
print(f"pooled RMSE (main rows): baseline {np.sqrt((main.resid_base ** 2).mean()):.2f} K, "
      f"benchmark {np.sqrt((main.resid_bench ** 2).mean()):.2f} K")
print()
print("By group:")
print(by_group.to_string())
print()
print("By LTST hour (main):")
print(by_hour.to_string())
print()
print("By Ls bin (main):")
print(by_ls[["n", "baseline_bias_k", "bench_bias_k"]].to_string())

tbl.to_csv(OUT / "lobo_rows.csv", index=False)
by_block.to_csv(OUT / "lobo_by_block.csv")
by_group.to_csv(OUT / "lobo_by_group.csv")
by_hour.to_csv(OUT / "lobo_by_ltst_hour.csv")
by_ls.to_csv(OUT / "lobo_by_ls_bin.csv")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ax = axes[0]
x = np.arange(len(by_block))
ax.bar(x - 0.2, by_block.baseline_rmse_k, 0.4, label="baseline")
ax.bar(x + 0.2, by_block.bench_rmse_k, 0.4, label="benchmark")
ax.set_xticks(x, [str(b) for b in by_block.index])
ax.set(xlabel="held-out block", ylabel="RMSE (K)", title="Leave-one-block-out RMSE")
ax.legend(fontsize=8)
ax = axes[1]
ax.plot(by_hour.index, by_hour.baseline_bias_k, "o-", ms=3, label="baseline bias")
ax.plot(by_hour.index, by_hour.bench_bias_k, "s-", ms=3, label="benchmark bias")
ax.plot(by_hour.index, by_hour.baseline_rmse_k, "--", label="baseline RMSE")
ax.plot(by_hour.index, by_hour.bench_rmse_k, ":", label="benchmark RMSE")
ax.axhline(0, color="grey", lw=0.8)
ax.set(xlabel="LTST hour", ylabel="K", title="By LTST hour (main rows)")
ax.legend(fontsize=8)
ax = axes[2]
ax.bar(by_ls.index + LS_BIN_DEG / 2 - 4, by_ls.baseline_bias_k, 8, label="baseline")
ax.bar(by_ls.index + LS_BIN_DEG / 2 + 4, by_ls.bench_bias_k, 8, label="benchmark")
ax.axhline(0, color="grey", lw=0.8)
ax.set(xlabel="Ls (deg)", ylabel="bias (K)", title="Bias by 30-degree Ls bin (main rows)")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "lobo_scores.png", dpi=120)

fig, axes = plt.subplots(3, 3, figsize=(13, 9), sharex=True, sharey=True)
for ax, b in zip(axes.flat, blocks):
    g = tbl[tbl.block == b].groupby("ltst_hour")[["ground_temp_k", "baseline_k", "bench_k"]].mean()
    ax.plot(g.index, g.ground_temp_k, "k.-", label="REMS")
    ax.plot(g.index, g.baseline_k, "r.-", label="baseline (fold fit)")
    ax.plot(g.index, g.bench_k, "b.--", label="benchmark")
    s = tbl[tbl.block == b].sol
    ax.set_title(f"block {b}: sols {s.min()}-{s.max()} (held out)", fontsize=9)
axes.flat[0].legend(fontsize=7)
fig.supxlabel("LTST hour")
fig.supylabel("ground temperature (K)")
fig.tight_layout()
fig.savefig(OUT / "lobo_diurnal_by_block.png", dpi=120)
print("saved to", OUT)